# ARC-05: Classical Fourth-Order Runge-Kutta Integrator

For y' = f(t, y), one RK4 step is:

- k1 = f(t, y)
- k2 = f(t + dt/2, y + dt*k1/2)
- k3 = f(t + dt/2, y + dt*k2/2)
- k4 = f(t + dt, y + dt*k3)
- y_next = y + dt*(k1 + 2*k2 + 2*k3 + k4)/6

The integrator is deterministic, float64-based, and does not mutate its input state.


In [1]:
import math
import numpy as np


def rk4_step(state, time_s, time_step_s, derivative, *derivative_args):
    if isinstance(time_step_s, (bool, np.bool_, str, bytes, bytearray)):
        raise ValueError("Timestep must be a finite numeric scalar")

    try:
        dt = float(time_step_s)
        t = float(time_s)
    except (TypeError, ValueError, OverflowError) as exc:
        raise ValueError("Time and timestep must be numeric scalars") from exc

    if isinstance(time_s, (bool, np.bool_, str, bytes, bytearray)):
        raise ValueError("Time must be a finite numeric scalar")
    if not math.isfinite(dt) or dt <= 0:
        raise ValueError("Timestep must be finite and positive")
    if not math.isfinite(t):
        raise ValueError("Time must be finite")
    if t < 0.0:
        raise ValueError("Time must be non-negative")

    if isinstance(state, (str, bytes, bytearray, bool, np.bool_)):
        raise ValueError("State must be a numeric one-dimensional array")

    try:
        components = list(state)
    except (TypeError, ValueError, OverflowError) as exc:
        raise ValueError("State must be a numeric one-dimensional array") from exc

    for component in components:
        if isinstance(component, (bool, np.bool_, str, bytes, bytearray, complex, np.complexfloating, list, tuple, np.ndarray)):
            raise ValueError("State must contain only real numeric scalar values")

    try:
        raw = np.asarray(state)
    except (TypeError, ValueError, OverflowError) as exc:
        raise ValueError("State must be a numeric one-dimensional array") from exc

    if raw.dtype.kind not in "iuf" or raw.ndim != 1:
        raise ValueError("State must contain only real numeric values in one dimension")

    y = np.asarray(raw, dtype=np.float64).copy()
    if y.size == 0 or not np.all(np.isfinite(y)):
        raise ValueError("State must be non-empty and finite")
    if not callable(derivative):
        raise ValueError("Derivative must be callable")

    k1 = np.asarray(derivative(t, y, *derivative_args), dtype=np.float64)
    k2 = np.asarray(derivative(t + 0.5 * dt, y + 0.5 * dt * k1, *derivative_args), dtype=np.float64)
    k3 = np.asarray(derivative(t + 0.5 * dt, y + 0.5 * dt * k2, *derivative_args), dtype=np.float64)
    k4 = np.asarray(derivative(t + dt, y + dt * k3, *derivative_args), dtype=np.float64)

    for k_name, k in (("k1", k1), ("k2", k2), ("k3", k3), ("k4", k4)):
        if k.shape != y.shape:
            raise ValueError(f"{k_name} has the wrong shape")
        if not np.all(np.isfinite(k)):
            raise ValueError(f"{k_name} contains non-finite values")

    next_state = y + (dt / 6.0) * (k1 + 2.0 * k2 + 2.0 * k3 + k4)
    if next_state.shape != y.shape or not np.all(np.isfinite(next_state)):
        raise ValueError("RK4 produced an invalid state")

    return next_state


def linear_derivative(time_s, state, slope):
    if isinstance(slope, (bool, np.bool_, str, bytes, bytearray)):
        raise ValueError("Slope must be finite and numeric")
    try:
        slope = float(slope)
    except (TypeError, ValueError, OverflowError) as exc:
        raise ValueError("Slope must be finite and numeric") from exc
    if not np.isfinite(slope):
        raise ValueError("Slope must be finite and numeric")
    return slope * state


In [2]:
slope = 1.0
initial = np.array([1.0], dtype=np.float64)

# Analytic exponential reference: y(1) = exp(1).
for steps in (1, 10, 100):
    y = initial.copy()
    dt = 1.0 / steps
    for index in range(steps):
        y = rk4_step(y, index * dt, dt, linear_derivative, slope)
    print(f"steps={steps:3d} value={y[0]:.12f}")

dt_values = np.array([0.2, 0.1, 0.05, 0.025], dtype=np.float64)
errors = []
for dt in dt_values:
    steps = round(1.0 / dt)
    y = initial.copy()
    for index in range(steps):
        y = rk4_step(y, index * dt, 1.0 / steps, linear_derivative, slope)
    errors.append(abs(y[0] - math.e))

measured_ratios = [errors[i] / errors[i + 1] for i in range(len(errors) - 1)]
print("measured error ratios for successive 2x refinements:", measured_ratios)
assert all(current > smaller for current, smaller in zip(errors, errors[1:]))
# At finite steps these ratios approach 16 for fourth-order RK4, but rounding,
# truncation, and the sampled step sizes keep them away from exactly 16.
assert all(12.0 < ratio < 32.0 for ratio in measured_ratios)

original = initial.copy()
result = rk4_step(original, 0.0, 0.1, linear_derivative, slope)
assert np.array_equal(original, initial)
assert result.shape == initial.shape and np.all(np.isfinite(result))

for invalid_dt in (0.0, -1.0, float("nan"), float("inf"), True, "0.1"):
    try:
        rk4_step(initial, 0.0, invalid_dt, linear_derivative, slope)
        raise AssertionError("Invalid timestep accepted")
    except ValueError:
        pass

for invalid_state in ([], np.array([float("nan")]), np.array([1 + 0j]), [[1.0]], [True], ["1"]):
    try:
        rk4_step(invalid_state, 0.0, 0.1, linear_derivative, slope)
        raise AssertionError("Invalid state accepted")
    except ValueError:
        pass

print("RK4 analytic convergence checks: PASSED")
print("RK4 invalid-input checks: PASSED")


steps=  1 value=2.708333333333
steps= 10 value=2.718279744135
steps=100 value=2.718281828234
measured error ratios for successive 2x refinements: [np.float64(14.725088271988833), np.float64(15.34817589271767), np.float64(15.6704045488986)]
RK4 analytic convergence checks: PASSED
RK4 invalid-input checks: PASSED
